<a href="https://colab.research.google.com/github/aexomir/semantic-correspondence/blob/step-1/env.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!test -d /content/semantic-correspondence || git clone -q https://github.com/aexomir/semantic-correspondence.git /content/semantic-correspondence
!pip -q install -U pip
!pip -q install -r /content/semantic-correspondence/requirements.txt
!pip -q install opencv-python tqdm
!pip -q install git+https://github.com/facebookresearch/segment-anything.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!rm -rf dinov2 dinov3
!git clone -q https://github.com/facebookresearch/dinov2.git
!git clone -q https://github.com/facebookresearch/dinov3.git

!pip -q install omegaconf torchmetrics fvcore iopath submitit ftfy regex scikit-learn termcolor

In [3]:
import os
import shutil
import torch
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

MYDRIVE = '/content/drive/MyDrive'

DINOV2_WEIGHTS = os.path.join(MYDRIVE, 'dinov2_vitb14_pretrain_small.pth')
DINOV3_WEIGHTS = os.path.join(MYDRIVE, 'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth')
SAM_WEIGHTS_B  = os.path.join(MYDRIVE, 'sam_vit_b_01ec64_small.pth')

SPAIR_DIR = os.path.join(MYDRIVE, 'SPair-71k')
SPAIR_TAR = os.path.join(MYDRIVE, 'SPair-71k.tar.gz')
PFW_ZIP   = os.path.join(MYDRIVE, 'pf-willow.zip')

LOCAL_DATA_DIR = '/content/data'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

if not os.path.isdir(MYDRIVE):
    raise RuntimeError('Drive not mounted. Complete the auth prompt and re-run this cell.')

print('OK:', MYDRIVE)


Mounted at /content/drive
OK: /content/drive/MyDrive


In [4]:
local_spair = os.path.join(LOCAL_DATA_DIR, 'SPair-71k')

if os.path.isdir(local_spair):
    print('OK:', local_spair)
elif os.path.isfile(SPAIR_TAR):
    shutil.unpack_archive(SPAIR_TAR, LOCAL_DATA_DIR, format='gztar')
    print('OK:', local_spair)
elif os.path.isdir(SPAIR_DIR):
    raise RuntimeError('Found SPair-71k as a Drive folder. Put SPair-71k.tar.gz in MyDrive and re-run (faster + avoids Drive timeouts).')
else:
    raise FileNotFoundError('Missing SPair-71k.tar.gz in MyDrive.')


OK: /content/data/SPair-71k


In [5]:
from segment_anything import SamPredictor, sam_model_registry

# SAM ViT-B
sam = sam_model_registry['vit_b'](checkpoint=SAM_WEIGHTS_B)
sam_predictor = SamPredictor(sam)

# DINOv2 and DINOv3 from local clones
# We load architecture from repo and then load weights from Drive.
dinov2 = torch.hub.load('dinov2', 'dinov2_vitb14', source='local', pretrained=False)
dinov2.load_state_dict(torch.load(DINOV2_WEIGHTS, map_location='cpu'))

dinov3 = torch.hub.load('dinov3', 'dinov3_vitb16', source='local', pretrained=False)
dinov3.load_state_dict(torch.load(DINOV3_WEIGHTS, map_location='cpu'))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dinov2.to(device).eval()
dinov3.to(device).eval()
sam.to(device).eval()

print('Ready. Device:', device)


/content/dinov2/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/content/dinov2/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/content/dinov2/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Ready. Device: cpu


In [6]:
import json
import math
from contextlib import nullcontext

import cv2
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torchvision import transforms

def extract_dino_features(model, img_tensor):
    """Return dense patch tokens as (1, D, H, W)."""
    model.eval()
    with torch.no_grad():
        out = model.forward_features(img_tensor)
        if isinstance(out, dict):
            patch_tokens = out.get("x_norm_patchtokens", out.get("x_norm_patch_tokens"))
        else:
            patch_tokens = out

        if patch_tokens is None:
            keys = list(out.keys()) if isinstance(out, dict) else "N/A"
            raise ValueError(f"Could not find patch tokens. Keys: {keys}")

        b, n, d = patch_tokens.shape
        grid = int(math.isqrt(n))
        if grid * grid != n:
            raise ValueError(f"Expected square number of patches, got N={n}")

        return patch_tokens.permute(0, 2, 1).reshape(b, d, grid, grid)


def extract_dino_layers(model, img_tensor, layer_ids):
    """Return dict {layer_id: (1, D, H, W)}."""
    model.eval()
    with torch.no_grad():
        all_layers = model.get_intermediate_layers(img_tensor, n=len(model.blocks), norm=True)
        out = {}
        for lid in layer_ids:
            patch_tokens = all_layers[lid]
            b, n, d = patch_tokens.shape
            grid = int(math.isqrt(n))
            if grid * grid != n:
                raise ValueError(f"Layer {lid}: expected square patches, got N={n}")
            out[lid] = patch_tokens.permute(0, 2, 1).reshape(b, d, grid, grid)
        return out


def extract_sam_features(predictor, image_np, res=512):
    """Return SAM image-encoder features as (1, C, H, W) for an input resized to res x res."""
    device = next(predictor.model.parameters()).device
    encoder = predictor.model.image_encoder

    image_resized = cv2.resize(image_np, (res, res), interpolation=cv2.INTER_LINEAR)

    img_tensor = torch.from_numpy(image_resized).float().permute(2, 0, 1).unsqueeze(0).to(device)

    pixel_mean = torch.tensor([123.675, 116.28, 103.53], device=device).view(-1, 1, 1)
    pixel_std = torch.tensor([58.395, 57.12, 57.375], device=device).view(-1, 1, 1)
    img_tensor = (img_tensor - pixel_mean) / pixel_std

    with torch.no_grad():
        orig_pos_embed = encoder.pos_embed
        new_size = res // 16  # SAM patch size

        if orig_pos_embed is not None and orig_pos_embed.shape[1] != new_size:
            pos_embed_interp = F.interpolate(
                orig_pos_embed.data.permute(0, 3, 1, 2),
                size=(new_size, new_size),
                mode="bicubic",
                align_corners=False,
            ).permute(0, 2, 3, 1)

            encoder.pos_embed = torch.nn.Parameter(pos_embed_interp, requires_grad=False)
            feats = encoder(img_tensor)
            encoder.pos_embed = orig_pos_embed
        else:
            feats = encoder(img_tensor)

    return feats


In [7]:
SPAIR_ROOT = os.path.join(LOCAL_DATA_DIR, "SPair-71k")
JPEG_ROOT = os.path.join(SPAIR_ROOT, "JPEGImages")
PAIRANN_TEST_ROOT = os.path.join(SPAIR_ROOT, "PairAnnotation", "test")

FEATURE_ROOT = os.path.join(MYDRIVE, "features", "spair71k")
RESULTS_ROOT = os.path.join(MYDRIVE, "results")

MODEL_SPECS = {
    "dinov2_vitb14": {
        "model": dinov2,
        "img_size": (518, 518),
        "kind": "dino",
    },
    "dinov3_vitb16": {
        "model": dinov3,
        "img_size": (512, 512),
        "kind": "dino",
    },
    "sam_vit_b_res512": {
        "model": sam_predictor,
        "img_size": (512, 512),
        "kind": "sam",
        "sam_res": 512,
    },
}


def _ensure_parent_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)


def _rel_from_jpeg_root(image_path: str) -> str:
    rel = os.path.relpath(image_path, JPEG_ROOT)
    if rel.startswith(".."):
        raise ValueError(f"Expected image under JPEG_ROOT={JPEG_ROOT}, got {image_path}")
    return rel


def feature_path(model_key: str, image_path: str) -> str:
    rel = _rel_from_jpeg_root(image_path)
    rel_no_ext = os.path.splitext(rel)[0]
    return os.path.join(FEATURE_ROOT, model_key, rel_no_ext + ".npz")


def save_feature_npz(path: str, feat_np: np.ndarray) -> None:
    _ensure_parent_dir(path)
    np.savez_compressed(path, feat=feat_np)


def load_feature_npz(path: str) -> np.ndarray:
    with np.load(path) as z:
        return z["feat"]


def dino_transform(img_size):
    return transforms.Compose(
        [
            transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )


_DINO_TRANSFORMS = {
    k: dino_transform(v["img_size"])
    for k, v in MODEL_SPECS.items()
    if v["kind"] == "dino"
}


def compute_feature(model_key: str, image_path: str) -> np.ndarray:
    spec = MODEL_SPECS[model_key]
    kind = spec["kind"]

    img_pil = Image.open(image_path).convert("RGB")

    if kind == "dino":
        t = _DINO_TRANSFORMS[model_key](img_pil).unsqueeze(0).to(device)
        feat = extract_dino_features(spec["model"], t)  # (1, D, H, W)
    elif kind == "sam":
        img_np = np.array(img_pil)
        feat = extract_sam_features(spec["model"], img_np, res=spec["sam_res"])  # (1, C, H, W)
    else:
        raise ValueError(f"Unknown kind: {kind}")

    feat_np = feat.squeeze(0).detach().float().cpu().numpy().astype(np.float16)
    return feat_np


def load_or_compute_feature(model_key: str, image_path: str) -> np.ndarray:
    out_path = feature_path(model_key, image_path)
    if os.path.exists(out_path):
        return load_feature_npz(out_path)

    feat_np = compute_feature(model_key, image_path)
    save_feature_npz(out_path, feat_np)
    return feat_np


print("Cache root:", FEATURE_ROOT)
print("Results root:", RESULTS_ROOT)
print("JPEG root:", JPEG_ROOT)
print("PairAnnotation test root:", PAIRANN_TEST_ROOT)


Cache root: /content/drive/MyDrive/features/spair71k
Results root: /content/drive/MyDrive/results
JPEG root: /content/data/SPair-71k/JPEGImages
PairAnnotation test root: /content/data/SPair-71k/PairAnnotation/test


In [8]:
VALID_IMAGE_EXTS = {".jpg", ".jpeg", ".png"}


def iter_jpeg_images():
    for root, _, files in os.walk(JPEG_ROOT):
        for fn in files:
            ext = os.path.splitext(fn)[1].lower()
            if ext in VALID_IMAGE_EXTS:
                yield os.path.join(root, fn)


def precompute_features(model_keys=None, limit=None):
    if model_keys is None:
        model_keys = list(MODEL_SPECS.keys())

    images = list(iter_jpeg_images())
    if limit is not None:
        images = images[: int(limit)]

    print(f"Found {len(images)} images under {JPEG_ROOT}")
    print("Models:", model_keys)

    for model_key in model_keys:
        missing = 0
        for img_path in tqdm(images, desc=f"{model_key}: caching"):
            out_path = feature_path(model_key, img_path)
            if os.path.exists(out_path):
                continue

            feat_np = compute_feature(model_key, img_path)
            save_feature_npz(out_path, feat_np)
            missing += 1

        print(f"{model_key}: newly cached {missing} feature files")


# Example:
# precompute_features(["dinov2_vitb14"], limit=50)
# precompute_features(list(MODEL_SPECS.keys()))


In [9]:
PCK_THRESHOLDS = [0.05, 0.1, 0.15, 0.2]


def list_test_pair_files():
    if not os.path.isdir(PAIRANN_TEST_ROOT):
        raise FileNotFoundError(f"Missing PairAnnotation test dir: {PAIRANN_TEST_ROOT}")

    pair_files = []
    for root, _, files in os.walk(PAIRANN_TEST_ROOT):
        for fn in files:
            if fn.endswith(".json"):
                pair_files.append(os.path.join(root, fn))

    pair_files.sort()
    return pair_files


def _parse_xy(p):
    if p is None:
        return None
    if not isinstance(p, (list, tuple)):
        return None
    if len(p) < 2:
        return None
    x, y = float(p[0]), float(p[1])
    if x < 0 or y < 0:
        return None
    return x, y


def _normalize_feat_np(feat_np: np.ndarray) -> np.ndarray:
    feat = feat_np.astype(np.float32, copy=False)
    denom = np.linalg.norm(feat, axis=0, keepdims=True)
    denom = np.maximum(denom, 1e-12)
    return feat / denom


def softargmax_2d_numpy(sim_map: np.ndarray, temperature=0.01, window_size=5):
    """Window soft-argmax around peak. Returns (x, y) in feature-map coordinates."""
    h, w = sim_map.shape
    flat = int(np.argmax(sim_map))
    peak_y = flat // w
    peak_x = flat % w

    half = window_size // 2
    y0 = max(0, peak_y - half)
    y1 = min(h, peak_y + half + 1)
    x0 = max(0, peak_x - half)
    x1 = min(w, peak_x + half + 1)

    window = sim_map[y0:y1, x0:x1]
    wf = window.reshape(-1).astype(np.float64)

    wf = (wf - wf.max()) / float(temperature)
    probs = np.exp(wf)
    probs = probs / probs.sum()

    ys = np.arange(y0, y1, dtype=np.float64)
    xs = np.arange(x0, x1, dtype=np.float64)
    grid_y, grid_x = np.meshgrid(ys, xs, indexing="ij")

    grid_xf = grid_x.reshape(-1)
    grid_yf = grid_y.reshape(-1)

    pred_x = float((probs * grid_xf).sum())
    pred_y = float((probs * grid_yf).sum())
    return pred_x, pred_y


def evaluate_cached(
    model_key: str,
    use_softargmax: bool,
    softargmax_temp: float = 0.01,
    softargmax_window: int = 5,
    thresholds=PCK_THRESHOLDS,
    max_pairs=None,
    save_csv=True,
):
    pair_files = list_test_pair_files()
    if max_pairs is not None:
        pair_files = pair_files[: int(max_pairs)]

    correct_kps = {t: 0 for t in thresholds}
    total_kps = 0

    per_image_results = []

    for pair_path in tqdm(pair_files, desc=f"Evaluating {model_key}"):
        with open(pair_path, "r") as f:
            ann = json.load(f)

        category = ann["category"]
        src_name = ann["src_imname"]
        trg_name = ann["trg_imname"]

        src_img_path = os.path.join(JPEG_ROOT, category, src_name)
        trg_img_path = os.path.join(JPEG_ROOT, category, trg_name)

        src_pil = Image.open(src_img_path).convert("RGB")
        trg_pil = Image.open(trg_img_path).convert("RGB")
        src_w, src_h = src_pil.size
        trg_w, trg_h = trg_pil.size

        f_src = _normalize_feat_np(load_or_compute_feature(model_key, src_img_path))
        f_trg = _normalize_feat_np(load_or_compute_feature(model_key, trg_img_path))

        d, fh, fw = f_src.shape
        _, th, tw = f_trg.shape
        if (fh, fw) != (th, tw):
            raise ValueError(f"Feature grids mismatch: src={(fh, fw)} trg={(th, tw)}")

        src_kps = ann["src_kps"]
        trg_kps = ann["trg_kps"]

        trg_bbox = ann["trg_bndbox"]
        bbox_w = float(trg_bbox[2] - trg_bbox[0])
        bbox_h = float(trg_bbox[3] - trg_bbox[1])
        norm_factor = max(bbox_w, bbox_h)

        image_correct = {t: 0 for t in thresholds}
        image_total_kps = 0

        kp_indices = range(len(src_kps)) if isinstance(src_kps, list) else src_kps.keys()
        for idx in kp_indices:
            p_src = _parse_xy(src_kps[idx])
            p_trg = _parse_xy(trg_kps[idx])
            if p_src is None or p_trg is None:
                continue

            sx, sy = p_src
            tx, ty = p_trg

            feat_x = int(round(sx / src_w * (fw - 1)))
            feat_y = int(round(sy / src_h * (fh - 1)))
            feat_x = min(max(feat_x, 0), fw - 1)
            feat_y = min(max(feat_y, 0), fh - 1)

            target_feat = f_src[:, feat_y, feat_x]  # (D,)
            sim = np.einsum("d,dhw->hw", target_feat, f_trg)  # (H, W)

            if use_softargmax:
                pred_x_feat, pred_y_feat = softargmax_2d_numpy(
                    sim,
                    temperature=softargmax_temp,
                    window_size=softargmax_window,
                )
            else:
                flat = int(np.argmax(sim))
                pred_y_feat = float(flat // fw)
                pred_x_feat = float(flat % fw)

            pred_x = ((pred_x_feat + 0.5) / fw) * trg_w
            pred_y = ((pred_y_feat + 0.5) / fh) * trg_h

            dist = float(np.sqrt((pred_x - tx) ** 2 + (pred_y - ty) ** 2))

            total_kps += 1
            image_total_kps += 1

            for t in thresholds:
                if dist <= (t * norm_factor):
                    correct_kps[t] += 1
                    image_correct[t] += 1

        per_image_results.append(
            {
                "category": category,
                "src_image": src_name,
                "trg_image": trg_name,
                "total_kps": image_total_kps,
                **{
                    f"PCK@{t}": (image_correct[t] / image_total_kps * 100.0)
                    if image_total_kps > 0
                    else 0.0
                    for t in thresholds
                },
            }
        )

    per_keypoint = {
        f"PCK@{t}": (correct_kps[t] / total_kps * 100.0) if total_kps > 0 else 0.0
        for t in thresholds
    }

    results = {
        "model_key": model_key,
        "use_softargmax": bool(use_softargmax),
        "softargmax_temp": float(softargmax_temp),
        "softargmax_window": int(softargmax_window),
        "per_keypoint": per_keypoint,
        "per_image": per_image_results,
        "total_keypoints": int(total_kps),
        "total_image_pairs": int(len(per_image_results)),
    }

    print("\nPer-keypoint PCK:")
    for t in thresholds:
        print(f"  PCK@{t}: {per_keypoint[f'PCK@{t}']:.2f}%")

    if save_csv:
        os.makedirs(RESULTS_ROOT, exist_ok=True)
        suffix = "softargmax" if use_softargmax else "argmax"
        out_csv = os.path.join(RESULTS_ROOT, f"{model_key}-{suffix}_per_image_results.csv")
        pd.DataFrame(per_image_results).to_csv(out_csv, index=False)
        print("Saved per-image CSV:", out_csv)

    return results


# Example:
# results = evaluate_cached("dinov2_vitb14", use_softargmax=False, max_pairs=200)
# results = evaluate_cached("dinov2_vitb14", use_softargmax=True, softargmax_temp=0.01, softargmax_window=7, max_pairs=200)


In [10]:
precompute_features(list(MODEL_SPECS.keys()))

eval_summaries = []
for model_key in MODEL_SPECS.keys():
    res_argmax = evaluate_cached(
        model_key,
        use_softargmax=False,
        save_csv=True,
    )
    eval_summaries.append(
        {
            "model": model_key,
            "mode": "argmax",
            **res_argmax["per_keypoint"],
            "total_keypoints": res_argmax["total_keypoints"],
            "total_image_pairs": res_argmax["total_image_pairs"],
        }
    )

summary_df = pd.DataFrame(eval_summaries)
os.makedirs(RESULTS_ROOT, exist_ok=True)
summary_path = os.path.join(RESULTS_ROOT, "summary_per_keypoint_pck.csv")
summary_df.to_csv(summary_path, index=False)
print("\nSaved summary:", summary_path)
display(summary_df)

Found 1800 images under /content/data/SPair-71k/JPEGImages
Models: ['dinov2_vitb14', 'dinov3_vitb16', 'sam_vit_b_res512']


dinov2_vitb14: caching:   0%|          | 0/1800 [00:00<?, ?it/s]

dinov2_vitb14: newly cached 0 feature files


dinov3_vitb16: caching:   0%|          | 0/1800 [00:00<?, ?it/s]

dinov3_vitb16: newly cached 1113 feature files


...

Evaluating dinov3_vitb16:   0%|          | 0/12234 [00:00<?, ?it/s]


Per-keypoint PCK:
  PCK@0.05: 35.78%
  PCK@0.1: 53.91%
  PCK@0.15: 63.57%
  PCK@0.2: 69.50%
Saved per-image CSV: /content/drive/MyDrive/results/dinov3_vitb16-argmax_per_image_results.csv


Evaluating sam_vit_b_res512:   0%|          | 0/12234 [00:00<?, ?it/s]


Per-keypoint PCK:
  PCK@0.05: 13.15%
  PCK@0.1: 23.24%
  PCK@0.15: 31.02%
  PCK@0.2: 37.76%
Saved per-image CSV: /content/drive/MyDrive/results/sam_vit_b_res512-argmax_per_image_results.csv

Saved summary: /content/drive/MyDrive/results/summary_per_keypoint_pck.csv


,model,mode,PCK@0.05,PCK@0.1,PCK@0.15,PCK@0.2,total_keypoints,total_image_pairs
0,dinov2_vitb14,argmax,36.685989,54.225161,64.026130,70.737478,88328,12234
1,dinov3_vitb16,argmax,35.775745,53.914953,63.567612,69.503442,88328,12234
2,sam_vit_b_res512,argmax,13.149851,23.241781,31.024137,37.755865,88328,12234
